# Hotel Booking Demand - End-to-End Exploratory Data Analysis

This notebook is a professional portfolio EDA project using a real public dataset with
more than 20,000 records. It covers data loading, data cleaning, missing values,
duplicates, type correction, outlier detection, feature engineering, visualization,
business questions, and final conclusions.

## Dataset Source

- Source CSV: https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv
- Original Kaggle dataset: https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand
- Research paper DOI: https://doi.org/10.1016/j.dib.2018.11.126

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Dataset").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "scripts"))
import hotel_booking_eda as hbeda

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

print(f"Project root: {PROJECT_ROOT}")

## Step 1 - Load the Real Public Dataset

The CSV is stored locally in the `Dataset` folder so the notebook can run without
Kaggle authentication or live downloads.

In [ ]:
raw_df = hbeda.load_raw_data(PROJECT_ROOT / "Dataset" / "hotel_bookings.csv")
print(f"Raw shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")
raw_df.head()

## Step 2 - Initial Data Audit

Before cleaning, inspect missing values, duplicate rows, data types, and core numeric
summary statistics.

In [ ]:
audit_summary = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing_values": raw_df.isna().sum(),
    "missing_rate": raw_df.isna().mean().round(4),
    "unique_values": raw_df.nunique()
}).sort_values("missing_values", ascending=False)

print(f"Exact duplicate rows: {raw_df.duplicated().sum():,}")
audit_summary.head(12)

In [ ]:
raw_df.describe(include="all").transpose().head(20)

## Step 3 - Data Cleaning and Feature Engineering

The reusable project function performs the full cleaning pipeline:

- Fills missing identifier/category values.
- Removes exact duplicates.
- Converts dates and corrects data types.
- Removes invalid zero-guest, zero-night, negative-ADR, and invalid-date rows.
- Engineers business features for stay length, guests, seasonality, revenue proxy,
  room changes, family bookings, and segmentation.

In [ ]:
clean_df, cleaning_summary, outlier_summary, missing_before = hbeda.clean_data(raw_df)
clean_df.to_csv(PROJECT_ROOT / "Dataset" / "hotel_bookings_cleaned.csv", index=False)

pd.DataFrame([cleaning_summary]).transpose().rename(columns={0: "value"})

## Step 4 - IQR Outlier Detection

Outliers are detected using the standard IQR rule. They are flagged rather than blindly
deleted because extreme lead times, high ADR values, and long stays can represent real
hotel business events.

In [ ]:
outlier_summary

## Step 5 - Business Metrics

In [ ]:
metrics = hbeda.compute_business_metrics(clean_df, cleaning_summary)
business_questions = hbeda.build_business_questions(metrics)
pd.DataFrame(business_questions)

## Step 6 - Professional Visualizations

Each chart below is generated with Matplotlib and/or Seaborn, saved into the `Images`
folder, and interpreted from a business perspective.

### 1. Missing Values Before Cleaning

**Visualization type:** Bar Chart

**Business insight:** Company and agent identifiers account for most missing values, so they are treated as explicit 'No Company' and 'No Agent' categories instead of being dropped. Country and children have small missing counts and are imputed safely.

In [ ]:
plot_object = hbeda.plot_missing_values_bar(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "01_missing_values_bar.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 2. Dataset Size Before and After Cleaning

**Visualization type:** Bar Chart

**Business insight:** The project starts with 119,390 rows and removes 31,994 exact duplicates, leaving 86,638 clean rows.

In [ ]:
plot_object = hbeda.plot_duplicate_records_bar(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "02_duplicate_records_bar.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 3. Booking Cancellation Distribution

**Visualization type:** Count Plot

**Business insight:** The overall cancellation rate is 27.7%, so cancellation is a major operational and revenue-risk theme in the dataset.

In [ ]:
plot_object = hbeda.plot_cancellation_distribution(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "03_cancellation_distribution.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 4. Booking Volume by Hotel Type

**Visualization type:** Count Plot

**Business insight:** City Hotel has the larger booking base with 53,043 records, making it the primary volume driver.

In [ ]:
plot_object = hbeda.plot_hotel_type_distribution(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "04_hotel_type_distribution.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 5. Cancellation Rate by Hotel Type

**Visualization type:** Bar Chart

**Business insight:** City Hotel has the higher cancellation rate at 30.2%, which suggests different risk management strategies by hotel type.

In [ ]:
plot_object = hbeda.plot_cancellation_rate_by_hotel(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "05_cancellation_rate_by_hotel.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 6. Monthly Booking Trend

**Visualization type:** Line Chart

**Business insight:** The monthly trend shows demand moving unevenly through time, with visible peaks that can support staffing, pricing, and campaign timing decisions.

In [ ]:
plot_object = hbeda.plot_monthly_bookings_trend(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "06_monthly_bookings_trend.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 7. Cancellation Rate by Arrival Month

**Visualization type:** Line Chart

**Business insight:** August has the highest cancellation rate at 32.4%, making it a priority month for deposit, reminder, or overbooking policies.

In [ ]:
plot_object = hbeda.plot_monthly_cancellation_rate(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "07_monthly_cancellation_rate.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 8. Booking Volume by Calendar Month

**Visualization type:** Count Plot

**Business insight:** August is the strongest demand month with 11,194 bookings.

In [ ]:
plot_object = hbeda.plot_booking_volume_by_month(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "08_booking_volume_by_month.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 9. Bookings by Arrival Year

**Visualization type:** Bar Chart

**Business insight:** 2016 has the highest observed booking count. Because the dataset does not cover every month equally across all years, year comparisons should be interpreted as dataset-period trends.

In [ ]:
plot_object = hbeda.plot_bookings_by_year(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "09_bookings_by_year.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 10. Bookings by Market Segment

**Visualization type:** Bar Chart

**Business insight:** Online TA is the leading acquisition segment with 51,285 bookings.

In [ ]:
plot_object = hbeda.plot_market_segment_bookings(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "10_market_segment_bookings.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 11. Bookings by Distribution Channel

**Visualization type:** Bar Chart

**Business insight:** TA/TO is the main distribution channel with 68,651 bookings.

In [ ]:
plot_object = hbeda.plot_distribution_channel_bookings(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "11_distribution_channel_bookings.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 12. Customer Type Share

**Visualization type:** Pie Chart

**Business insight:** Transient customers represent the biggest share of demand, so retention and cancellation controls should be tailored to this group.

In [ ]:
plot_object = hbeda.plot_customer_type_pie(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "12_customer_type_pie.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 13. Top 10 Source Countries

**Visualization type:** Bar Chart

**Business insight:** PRT is the strongest source market with 26,864 bookings.

In [ ]:
plot_object = hbeda.plot_top_countries(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "13_top_countries.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 14. ADR Distribution After IQR Capping

**Visualization type:** Histogram and Distribution Plot

**Business insight:** ADR is right-skewed, so the project keeps the original values but uses IQR-capped ADR for readable visuals and robust revenue comparisons.

In [ ]:
plot_object = hbeda.plot_adr_histogram(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "14_adr_histogram.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 15. Lead Time Distribution After IQR Capping

**Visualization type:** Histogram and Distribution Plot

**Business insight:** Lead time is also right-skewed, showing that most bookings are made within a moderate window while a smaller group books far in advance.

In [ ]:
plot_object = hbeda.plot_lead_time_distribution(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "15_lead_time_distribution.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 16. ADR Box Plot by Hotel Type

**Visualization type:** Box Plot

**Business insight:** City Hotel has the higher median capped ADR at 105.7, indicating stronger pricing power.

In [ ]:
plot_object = hbeda.plot_adr_boxplot_by_hotel(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "16_adr_boxplot_by_hotel.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 17. Lead Time Box Plot by Cancellation Status

**Visualization type:** Box Plot

**Business insight:** Canceled bookings tend to have longer lead times, which makes lead time a useful early warning feature for cancellation-risk monitoring.

In [ ]:
plot_object = hbeda.plot_lead_time_boxplot_by_status(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "17_lead_time_boxplot_by_status.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 18. ADR Violin Plot by Customer Type

**Visualization type:** Violin Plot

**Business insight:** The violin plot shows how ADR distributions differ by customer type, helping separate high-volume customer groups from high-rate customer groups.

In [ ]:
plot_object = hbeda.plot_adr_violin_by_customer_type(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "18_adr_violin_by_customer_type.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 19. Distribution Plot of Length of Stay

**Visualization type:** Distribution Plot

**Business insight:** 1-2 nights is the most common stay-length category, which is useful for room inventory planning and housekeeping schedules.

In [ ]:
plot_object = hbeda.plot_total_nights_distribution(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "19_total_nights_distribution.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 20. Correlation Matrix Heatmap

**Visualization type:** Heatmap and Correlation Matrix

**Business insight:** The correlation matrix highlights directional relationships: special requests move opposite to cancellation, while lead time is positively associated with risk.

In [ ]:
plot_object = hbeda.plot_correlation_heatmap(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "20_correlation_heatmap.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 21. Pair Plot of Key Numeric Features

**Visualization type:** Pair Plot

**Business insight:** The pair plot shows feature interactions and confirms that canceled and non-canceled bookings overlap heavily, so business rules should combine several indicators.

In [ ]:
plot_object = hbeda.plot_pair_plot_numeric_features(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "21_pair_plot_numeric_features.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 22. Lead Time vs ADR by Cancellation Status

**Visualization type:** Scatter Plot

**Business insight:** The scatter plot shows cancellation status across pricing and lead-time space, surfacing long-lead bookings as a visible risk cluster.

In [ ]:
plot_object = hbeda.plot_lead_time_vs_adr_scatter(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "22_lead_time_vs_adr_scatter.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 23. Estimated Realized Room Revenue by Hotel Type

**Visualization type:** Bar Chart

**Business insight:** City Hotel produces the largest estimated realized room revenue at 12,112,961 ADR-units.

In [ ]:
plot_object = hbeda.plot_realized_revenue_by_hotel(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "23_realized_revenue_by_hotel.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 24. Top Reserved Room Types

**Visualization type:** Bar Chart

**Business insight:** Reserved room type A is the dominant room product, so availability and pricing decisions around this type carry high impact.

In [ ]:
plot_object = hbeda.plot_room_type_demand(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "24_room_type_demand.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 25. Cancellation Rate by Number of Special Requests

**Visualization type:** Bar Chart

**Business insight:** Cancellation rate declines as special requests increase, suggesting requests are a practical signal of guest commitment.

In [ ]:
plot_object = hbeda.plot_special_requests_vs_cancellation(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "25_special_requests_vs_cancellation.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

### 26. Seasonal Booking Patterns by Hotel Type

**Visualization type:** Grouped Bar Chart

**Business insight:** Summer is the busiest season, while Summer contributes the most estimated realized revenue.

In [ ]:
plot_object = hbeda.plot_seasonal_booking_patterns(raw_df, clean_df, cleaning_summary, outlier_summary)
image_path = PROJECT_ROOT / "Images" / "26_seasonal_booking_patterns.png"
plot_object.savefig(image_path, bbox_inches="tight")
plt.close("all")
display(Image(filename=str(image_path)))

## Step 7 - Final Business Questions

In [ ]:
pd.DataFrame(business_questions)

## Final Conclusion

The analysis shows that hotel cancellation risk is strongly connected to booking behavior,
channel mix, lead time, special requests, deposit policy, and customer type. The project
retains 86,638 clean records, uses robust
outlier handling, and translates exploratory findings into business actions.

Selected conclusions:

- City Hotel leads with 53,043 clean bookings.
- The cleaned dataset has an overall cancellation rate of 27.7%.
- City Hotel has the higher cancellation rate at 30.2%.
- City Hotel contributes the most estimated realized room revenue: 12,112,961 ADR-units.
- August has the highest booking volume with 11,194 bookings.
- August has the highest cancellation rate at 32.4%.
- 2016 has the most records with 41,967 bookings.
- Online TA is the largest segment with 51,285 bookings.
- TA/TO is the leading distribution channel with 68,651 bookings.
- Transient customers dominate with 71,366 bookings.
